# MARASON on public MS/MS libraries (Kaggle)

Settings → **Accelerator: GPU T4 x2** (or P100), **Internet: on**.

Use *Save Version → Save & Run All (Commit)* for long runs: it runs in the background for up to 12 h,
and everything under `/kaggle/working` (data and results) becomes the notebook output.

To continue in a new session, add that output as an input (*Add Input → Your Work → this notebook*)
and set `RESUME_FROM` below. Finished stages are skipped; an interrupted stage restarts.

In [ ]:
DATASET = "nplib1"   # "nplib1" (smallest, comparable to ICEBERG), "massbank" or "msnlib"
MAX_EPOCHS = 50      # repo configs use 200; lower it if a stage does not fit in the 12 h session
RESUME_FROM = ""     # e.g. "/kaggle/input/<this-notebook>/ms-pred"

import os
os.environ.update(DATASET=DATASET, MAX_EPOCHS=str(MAX_EPOCHS), RESUME_FROM=RESUME_FROM)

In [ ]:
%%bash
cd /kaggle/working
[ -d ms-pred ] || git clone -b claude/magical-knuth-oswt8t https://github.com/giamyy/ms-pred.git
cd ms-pred && git pull -q
nvidia-smi --query-gpu=name,memory.total --format=csv

Install the environment (~10 min, into `/tmp`, not kept between sessions).

In [ ]:
%%bash
cd /kaggle/working/ms-pred
bash run_scripts/marason/public/kaggle_run.sh $DATASET setup 2>&1 | tail -20

Download and convert the data, then run MAGMa. nplib1 takes minutes; msnlib can take hours.

In [ ]:
%%bash
cd /kaggle/working/ms-pred
bash run_scripts/marason/public/kaggle_run.sh $DATASET data 2>&1 | grep -v "it/s]" | tail -40
cat data/spec_datasets/$DATASET/audit.json

Train MARASON and evaluate on the test fold. Full log: `/kaggle/working/train_$DATASET.log`.

In [ ]:
%%bash
cd /kaggle/working/ms-pred
bash run_scripts/marason/public/kaggle_run.sh $DATASET train > /kaggle/working/train_$DATASET.log 2>&1
status=$?
grep -v "it/s]" /kaggle/working/train_$DATASET.log | tail -60
exit $status